# AT5 - Análise exploratória e visualização de dados

Este notebook realiza uma análise exploratória do arquivo `train.csv`.

O arquivo `train.csv` deve estar salvo na mesma pasta deste notebook.

A análise contempla:

1. carregamento do dataset;
2. quantidade de linhas e colunas;
3. tipo de dado de cada variável;
4. síntese estatística;
5. análise da variável de sobrevivência;
6. relação entre sexo e sobrevivência;
7. distribuição das idades dos passageiros.


In [ ]:
# Importação das bibliotecas

import pandas as pd
import matplotlib.pyplot as plt

# Configuração para exibir mais colunas no notebook
pd.set_option("display.max_columns", None)


## 1. Carregamento do arquivo train.csv

O arquivo será carregado diretamente da mesma pasta do notebook.


In [ ]:
# Carregamento do dataset

df = pd.read_csv("train.csv")

# Visualização das primeiras linhas
df.head()


## 2. Quantidade de linhas e colunas do dataset


In [ ]:
# Quantidade de linhas e colunas

linhas, colunas = df.shape

print(f"Quantidade de linhas: {linhas}")
print(f"Quantidade de colunas: {colunas}")


## 3. Tipo de dado de cada variável


In [ ]:
# Tipos de dados das variáveis

tipos_dados = df.dtypes.reset_index()
tipos_dados.columns = ["Variável", "Tipo de dado"]

tipos_dados


## 4. Síntese estatística das variáveis

A primeira tabela apresenta as variáveis numéricas. A segunda tabela apresenta as variáveis categóricas.


In [ ]:
# Síntese estatística das variáveis numéricas

df.describe()


In [ ]:
# Síntese estatística das variáveis categóricas

df.describe(include="object")


## 5. Tabela de frequência e gráfico para a variável de sobrevivência

Como a variável de sobrevivência é categórica/binária, o gráfico de barras foi escolhido porque facilita a comparação entre as categorias.

No dataset Titanic, a variável costuma aparecer como `Survived`, com os valores:

- 0 = não sobreviveu;
- 1 = sobreviveu.

O código abaixo reconhece automaticamente `survival`, `Survived` ou `survived`.


In [ ]:
# Identificação da variável de sobrevivência

possiveis_nomes_survival = ["survival", "Survived", "survived"]

variavel_survival = None

for nome in possiveis_nomes_survival:
    if nome in df.columns:
        variavel_survival = nome
        break

if variavel_survival is None:
    raise ValueError("A variável de sobrevivência não foi encontrada no dataset.")

print(f"Variável de sobrevivência encontrada: {variavel_survival}")


In [ ]:
# Tabela de frequência da variável de sobrevivência

freq_survival = df[variavel_survival].value_counts().sort_index().reset_index()
freq_survival.columns = ["Sobrevivência", "Frequência"]

freq_survival["Descrição"] = freq_survival["Sobrevivência"].map({
    0: "Não sobreviveu",
    1: "Sobreviveu"
})

freq_survival


In [ ]:
# Gráfico de barras da variável de sobrevivência

freq_survival_grafico = df[variavel_survival].map({
    0: "Não sobreviveu",
    1: "Sobreviveu"
}).value_counts()

plt.figure(figsize=(7, 5))
freq_survival_grafico.plot(kind="bar")

plt.title("Frequência de sobrevivência dos passageiros")
plt.xlabel("Situação")
plt.ylabel("Quantidade de passageiros")
plt.xticks(rotation=0)
plt.show()


## 6. Quantas pessoas sobreviveram? Qual a classe mais frequente?


In [ ]:
# Quantidade de pessoas que sobreviveram

quantidade_sobreviventes = (df[variavel_survival] == 1).sum()

print(f"Quantidade de pessoas que sobreviveram: {quantidade_sobreviventes}")


In [ ]:
# Classe mais frequente

if "Pclass" in df.columns:
    classe_mais_frequente = df["Pclass"].mode()[0]
    quantidade_classe = (df["Pclass"] == classe_mais_frequente).sum()

    print(f"A classe mais frequente é: {classe_mais_frequente}")
    print(f"Quantidade de passageiros nessa classe: {quantidade_classe}")
else:
    print("A variável Pclass não foi encontrada no dataset.")


## 7. Relação entre sexo e sobrevivência

Nesta etapa, será analisado se existe diferença na sobrevivência entre homens e mulheres.


In [ ]:
# Identificação da variável sexo

possiveis_nomes_sex = ["sex", "Sex"]

variavel_sex = None

for nome in possiveis_nomes_sex:
    if nome in df.columns:
        variavel_sex = nome
        break

if variavel_sex is None:
    raise ValueError("A variável sexo não foi encontrada no dataset.")

print(f"Variável sexo encontrada: {variavel_sex}")


In [ ]:
# Tabela de frequência entre sexo e sobrevivência

tabela_sex_survival = pd.crosstab(
    df[variavel_sex],
    df[variavel_survival],
    margins=True
)

tabela_sex_survival.columns = ["Não sobreviveu", "Sobreviveu", "Total"]

tabela_sex_survival


In [ ]:
# Tabela percentual por sexo

tabela_percentual_sex_survival = pd.crosstab(
    df[variavel_sex],
    df[variavel_survival],
    normalize="index"
) * 100

tabela_percentual_sex_survival.columns = ["Não sobreviveu (%)", "Sobreviveu (%)"]

tabela_percentual_sex_survival.round(2)


In [ ]:
# Gráfico da relação entre sexo e sobrevivência

tabela_grafico = pd.crosstab(
    df[variavel_sex],
    df[variavel_survival]
)

tabela_grafico.columns = ["Não sobreviveu", "Sobreviveu"]

tabela_grafico.plot(kind="bar", figsize=(8, 5))

plt.title("Relação entre sexo e sobrevivência")
plt.xlabel("Sexo")
plt.ylabel("Quantidade de passageiros")
plt.xticks(rotation=0)
plt.legend(title="Situação")
plt.show()


In [ ]:
# Interpretação: mulheres sobreviveram mais que homens?

taxa_sobrevivencia_por_sexo = df.groupby(variavel_sex)[variavel_survival].mean() * 100

display(taxa_sobrevivencia_por_sexo.round(2))

if "female" in taxa_sobrevivencia_por_sexo.index and "male" in taxa_sobrevivencia_por_sexo.index:
    taxa_mulheres = taxa_sobrevivencia_por_sexo["female"]
    taxa_homens = taxa_sobrevivencia_por_sexo["male"]

    if taxa_mulheres > taxa_homens:
        print("Sim. A taxa de sobrevivência das mulheres foi maior que a dos homens.")
    elif taxa_mulheres < taxa_homens:
        print("Não. A taxa de sobrevivência dos homens foi maior que a das mulheres.")
    else:
        print("A taxa de sobrevivência foi igual entre homens e mulheres.")
else:
    print("Verifique os nomes das categorias da variável sexo no dataset.")


## 8. Distribuição das idades dos passageiros

Para analisar a idade, será usado um histograma. Esse tipo de gráfico é adequado para variáveis numéricas contínuas, pois mostra a concentração dos valores em intervalos.


In [ ]:
# Identificação da variável idade

possiveis_nomes_age = ["age", "Age"]

variavel_age = None

for nome in possiveis_nomes_age:
    if nome in df.columns:
        variavel_age = nome
        break

if variavel_age is None:
    raise ValueError("A variável idade não foi encontrada no dataset.")

print(f"Variável idade encontrada: {variavel_age}")


In [ ]:
# Estatísticas da idade

df[variavel_age].describe()


In [ ]:
# Quantidade de valores ausentes na idade

valores_ausentes_idade = df[variavel_age].isna().sum()

print(f"Quantidade de valores ausentes na variável idade: {valores_ausentes_idade}")


In [ ]:
# Histograma da idade

plt.figure(figsize=(8, 5))

df[variavel_age].dropna().plot(kind="hist", bins=20)

plt.title("Distribuição das idades dos passageiros")
plt.xlabel("Idade")
plt.ylabel("Frequência")
plt.show()


In [ ]:
# Interpretação automática da distribuição das idades

media_idade = df[variavel_age].mean()
mediana_idade = df[variavel_age].median()

print(f"Média da idade: {media_idade:.2f}")
print(f"Mediana da idade: {mediana_idade:.2f}")

if media_idade > mediana_idade:
    print("A distribuição apresenta assimetria à direita, pois a média é maior que a mediana.")
elif media_idade < mediana_idade:
    print("A distribuição apresenta assimetria à esquerda, pois a média é menor que a mediana.")
else:
    print("A distribuição parece aproximadamente simétrica, pois média e mediana são muito próximas.")

if mediana_idade < 40:
    print("A maioria dos passageiros tende a ser jovem ou adulta jovem.")
else:
    print("A maioria dos passageiros tende a ser adulta mais velha ou idosa.")


## Conclusão orientativa

Após executar o notebook, observe os resultados impressos e os gráficos.

Exemplo de interpretação esperada para o dataset Titanic:

A variável de sobrevivência pode ser analisada por gráfico de barras porque possui poucas categorias. A classe mais frequente geralmente é a terceira classe. Na relação entre sexo e sobrevivência, costuma-se observar maior taxa de sobrevivência entre mulheres. A idade tende a se concentrar em passageiros jovens e adultos jovens, com menor quantidade de pessoas idosas.
